In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load files into memory (these variables will be reused later)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Ensure numeric 'value' columns where applicable
for df, col in [(metric_df, 'value'), (trace_df, 'value'), (log_df, 'value')]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        # create a datetime column with tz conversion
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) File-level summary: total row count and distinct cmdb_id values (up to 50)
def file_summary_row(df, filename):
    row_count = df.shape[0]
    if 'cmdb_id' in df.columns:
        distinct_cmdb = df['cmdb_id'].dropna().unique().tolist()
    else:
        distinct_cmdb = []
    # limit to 50 entries for display
    distinct_str = ", ".join(map(str, distinct_cmdb[:50]))
    return {"file": filename, "row_count": row_count, "distinct_cmdb_ids_preview": distinct_str, "distinct_count": len(distinct_cmdb)}

file_summaries = pd.DataFrame([
    file_summary_row(metric_df, "metric.csv"),
    file_summary_row(trace_df, "trace.csv"),
    file_summary_row(log_df, "log.csv"),
    file_summary_row(error_logs_df, "error_logs.csv"),
])

# 2) metric.csv: top 20 (cmdb_id, kpi_name) pairs by count_points with global P95 of value
metric_group = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count_points=('value', 'size'),
    global_P95_of_value=('value', lambda v: float(np.nanpercentile(v.values, 95)) if v.dropna().size>0 else np.nan)
)
metric_top20 = metric_group.sort_values('count_points', ascending=False).head(20).reset_index(drop=True)

# 3) trace.csv: top 20 (cmdb_id, trace_name) pairs by count_points with global P95 of value
trace_group = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count_points=('value', 'size'),
    global_P95_of_value=('value', lambda v: float(np.nanpercentile(v.values, 95)) if v.dropna().size>0 else np.nan)
)
trace_top20 = trace_group.sort_values('count_points', ascending=False).head(20).reset_index(drop=True)

# 4) log.csv: top 20 (cmdb_id, log_name) pairs by count_points with global P95 of value
log_group = log_df.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    count_points=('value', 'size'),
    global_P95_of_value=('value', lambda v: float(np.nanpercentile(v.values, 95)) if v.dropna().size>0 else np.nan)
)
log_top20 = log_group.sort_values('count_points', ascending=False).head(20).reset_index(drop=True)

# 5) error_logs.csv: top 20 cmdb_id by count with earliest and latest timestamps (as UTC+8 datetimes)
if 'cmdb_id' in error_logs_df.columns:
    err_group = error_logs_df.groupby('cmdb_id', as_index=False).agg(
        count_entries=('timestamp', 'size'),
        earliest_timestamp_sec=('timestamp', 'min'),
        latest_timestamp_sec=('timestamp', 'max')
    )
    # convert earliest/latest to tz-aware datetimes (Asia/Shanghai)
    err_group['earliest_timestamp'] = pd.to_datetime(err_group['earliest_timestamp_sec'], unit='s', utc=True).dt.tz_convert(tz)
    err_group['latest_timestamp'] = pd.to_datetime(err_group['latest_timestamp_sec'], unit='s', utc=True).dt.tz_convert(tz)
    error_top20 = err_group.sort_values('count_entries', ascending=False).head(20)[
        ['cmdb_id', 'count_entries', 'earliest_timestamp', 'latest_timestamp']
    ].reset_index(drop=True)
else:
    error_top20 = pd.DataFrame(columns=['cmdb_id', 'count_entries', 'earliest_timestamp', 'latest_timestamp'])

# Keep outputs compact: show file summary and the top-20 tables (selected columns)
file_summaries, metric_top20, trace_top20, log_top20, error_top20

```
Out[1]:
```
```python
# Reuse loaded variables from the kernel: file_summaries, metric_top20, trace_top20, log_top20, error_top20
fs = file_summaries.set_index('file')

metric_rows = int(fs.loc['metric.csv', 'row_count'])
metric_cmdb_preview = fs.loc['metric.csv', 'distinct_cmdb_ids_preview']
metric_distinct_count = int(fs.loc['metric.csv', 'distinct_count'])

trace_rows = int(fs.loc['trace.csv', 'row_count'])
trace_cmdb_preview = fs.loc['trace.csv', 'distinct_cmdb_ids_preview']
trace_distinct_count = int(fs.loc['trace.csv', 'distinct_count'])

log_rows = int(fs.loc['log.csv', 'row_count'])
log_cmdb_preview = fs.loc['log.csv', 'distinct_cmdb_ids_preview']
log_distinct_count = int(fs.loc['log.csv', 'distinct_count'])

error_rows = int(fs.loc['error_logs.csv', 'row_count'])

# Top-summaries from top20 tables
metric_top_cmdbs = ", ".join(map(str, metric_top20['cmdb_id'].unique()[:6]))
metric_max_count = int(metric_top20['count_points'].max()) if not metric_top20.empty else 0
metric_pairs_shown = metric_top20.shape[0]

trace_top_cmdbs = ", ".join(map(str, trace_top20['cmdb_id'].unique()[:6]))
trace_max_count = int(trace_top20['count_points'].max()) if not trace_top20.empty else 0
trace_pairs_shown = trace_top20.shape[0]

log_top_cmdbs = ", ".join(map(str, log_top20['cmdb_id'].unique()[:6]))
log_max_count = int(log_top20['count_points'].max()) if not log_top20.empty else 0
log_pairs_shown = log_top20.shape[0]

error_present = not error_top20.empty

summary = (
    "Telemetry load summary:\n"
    f"- metric.csv: {metric_rows} rows, {metric_distinct_count} distinct cmdb_id values "
    f"(preview: {metric_cmdb_preview}). Top {metric_pairs_shown} (cmdb_id,kpi_name) pairs shown; "
    f"highest point count in that top set = {metric_max_count}. Primary cmdb_ids in top metrics: {metric_top_cmdbs}.\n\n"
    f"- trace.csv: {trace_rows} rows, {trace_distinct_count} distinct cmdb_id values "
    f"(preview: {trace_cmdb_preview}). Top {trace_pairs_shown} (cmdb_id,trace_name) pairs shown; "
    f"highest point count = {trace_max_count} (many trace pairs are single-point entries). Primary cmdb_ids in top traces: {trace_top_cmdbs}.\n\n"
    f"- log.csv: {log_rows} rows, {log_distinct_count} distinct cmdb_id values "
    f"(preview: {log_cmdb_preview}). Top {log_pairs_shown} (cmdb_id,log_name) pairs shown; "
    f"highest point count = {log_max_count}. Primary cmdb_ids in top logs: {log_top_cmdbs}. "
    "Common log names with high counts include log.row_count and log.error_count (error_count P95 mostly 0).\n\n"
    f"- error_logs.csv: {error_rows} rows. No error log entries present." 
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count                          distinct_cmdb_ids_preview  distinct_count
0      metric.csv      22525  IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redi...              25
1       trace.csv        264  IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, To...              12
2         log.csv        489  Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat0...               8
3  error_logs.csv          0                                                                  0,      cmdb_id                                           kpi_name  count_points  global_P95_of_value
0   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31         1.390000e+02
1   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31         2.304870e+05
2   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...            31         9.782726e+08
3   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...            31         1.886744e+06
4   Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter            31         0.000000e+00
5   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...            31         0.000000e+00
6   Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31         7.692402e+06
7   Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31         1.000000e+00
8   Tomcat04         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31         3.000000e+00
9       MG01           JVM-Threads_7779_JVM_ThreadCount_Threads            31         7.600000e+01
10      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31         2.040136e+09
11      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31         9.710172e+08
12      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31         5.907150e+07
13  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31         0.000000e+00
14  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31         0.000000e+00
15  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31         1.000000e+02
16  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31         4.000000e+00
17  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31         5.000000e+02
18  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter            31         0.000000e+00
19  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions            31         0.000000e+00,      cmdb_id                       trace_name  count_points  global_P95_of_value
0   dockerB2          trace.to_MG02.row_count             1          1740.000000
1       IG01    trace.from_root.duration_mean             1             0.397872
2       IG01     trace.from_root.duration_p95             1             1.439100
3       IG01        trace.from_root.row_count             1            47.000000
4       IG01         trace.self.duration_mean             1             0.343170
5       IG01          trace.self.duration_p95             1             1.778000
6       IG01             trace.self.row_count             1         37213.000000
7       IG01  trace.to_Tomcat01.duration_mean             1             0.338199
8   dockerA2          trace.to_MG01.row_count             1          1187.000000
9   dockerA2      trace.to_MG02.duration_mean             1             0.060294
10  dockerA2       trace.to_MG02.duration_p95             1             0.322000
11  dockerA2          trace.to_MG02.row_count             1           951.000000
12  dockerB1    trace.from_MG01.duration_mean             1             0.299556
13  dockerB1     trace.from_MG01.duration_p95             1             2.077550
14  dockerB1        trace.from_MG01.row_count             1          7410.000000
15  dockerB1    trace.from_MG02.duration_mean             1             0.304236
16  dockerA2    trace.from_root.duration_mean             1             0.009814
17  dockerA2     trace.from_root.duration_p95             1             0.021350
18  dockerA2        trace.from_root.row_count             1           172.000000
19  dockerA2         trace.self.duration_mean             1             0.011819,      cmdb_id          log_name  count_points  global_P95_of_value
0   Tomcat02     log.row_count            31             398.0000
1   Tomcat02   log.error_count            31               0.0000
2   Tomcat01     log.row_count            31             388.0000
3   Tomcat01   log.error_count            31               0.0000
4   apache02   log.error_count            31               0.0000
5   apache01     log.row_count            31             770.0000
6   apache02     log.row_count            31             761.5000
7   apache01   log.error_count            31               0.0000
8   Tomcat03   log.error_count            31               0.0000
9   Tomcat04     log.row_count            31             390.0000
10  Tomcat04   log.error_count            31               0.0000
11  Tomcat03     log.row_count            31             366.5000
12   Mysql01     log.row_count            27              10.7000
13   Mysql01   log.error_count            27               0.0000
14  Tomcat04  log.gc_pause_max            16               0.0925
15  Tomcat03  log.gc_pause_max            15               0.0860
16  Tomcat01  log.gc_pause_max            15               0.0900
17  Tomcat02  log.gc_pause_max            15               0.1060
18   Mysql02     log.row_count             1               1.0000
19   Mysql02   log.error_count             1               0.0000, Empty DataFrame
Columns: [cmdb_id, count_entries, earliest_timestamp, latest_timestamp]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse metric_df from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure dt column (UTC+8) exists
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure dtype & timezone
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Components of interest
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Filter rows to components of interest for further processing
metric_df_filtered_components = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# Ensure numeric 'value'
metric_df_filtered_components['value'] = pd.to_numeric(metric_df_filtered_components['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (no time filter)
def p95_series(arr):
    a = np.asarray(arr.dropna())
    return float(np.nanpercentile(a, 95)) if a.size > 0 else np.nan

metric_p95_df = (
    metric_df_filtered_components
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(global_P95=('value', p95_series))
)

# 2) Define incident time window in UTC+8 and filter metric rows to that window
start_str = "2021-03-23 17:30:00"
end_str   = "2021-03-23 18:00:00"
start_dt = tz.localize(pd.to_datetime(start_str))
end_dt   = tz.localize(pd.to_datetime(end_str))

metric_window_df = metric_df_filtered_components[
    (metric_df_filtered_components['dt'] >= start_dt) &
    (metric_df_filtered_components['dt'] <= end_dt)
].copy()

# 3) Join global P95 and identify anomalies (value > global_P95)
metric_window_df = metric_window_df.merge(metric_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

metric_anomaly_points = metric_window_df[
    metric_window_df['global_P95'].notna() &
    metric_window_df['value'].notna() &
    (metric_window_df['value'] > metric_window_df['global_P95'])
].copy()

# Aggregate anomalies per pair
anomaly_agg_all = (
    metric_anomaly_points
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        global_P95 = ('global_P95', 'first'),
        count_anom_points = ('value', 'size'),
        earliest_anom_timestamp = ('dt', 'min'),
        max_value = ('value', 'max')
    )
)

# breach_ratio calculation
anomaly_agg_all['breach_ratio'] = np.where(
    anomaly_agg_all['global_P95'] == 0,
    np.inf,
    (anomaly_agg_all['max_value'] - anomaly_agg_all['global_P95']) / anomaly_agg_all['global_P95']
)

# 4) Top 20 by breach_ratio desc
anomaly_agg_top20 = anomaly_agg_all.sort_values('breach_ratio', ascending=False).head(20).reset_index(drop=True)

# Expose intermediate variables in kernel and show compact result
metric_p95_df, metric_window_df.shape, metric_anomaly_points.shape, anomaly_agg_all.shape, anomaly_agg_top20

```
Out[2]:
```
# Compose a plain-English summary string using kernel-stored results
metric_p95_count = metric_p95_df.shape[0]
window_rows = metric_window_df.shape[0]
anom_point_count = metric_anomaly_points.shape[0]
anom_pair_count = anomaly_agg_all.shape[0]
top20_df = anomaly_agg_top20

summary = (
    f"Metric anomaly detection summary (components restricted to the provided list):\n\n"
    f"- Baseline: computed global P95 for {metric_p95_count} (cmdb_id, kpi_name) pairs using each pair's full series.\n"
    f"- Time window: filtered metric points to 2021-03-23 17:30:00 → 2021-03-23 18:00:00 (UTC+8), yielding {window_rows} rows in-window.\n"
    f"- Anomalies: found {anom_point_count} data points exceeding their pair-wise global P95, across {anom_pair_count} distinct (cmdb_id, kpi_name) pairs.\n\n"
    f"Top findings (up to 20 pairs, ranked by breach severity):\n"
    f"- Several pairs show 'infinite' breach_ratio because global_P95 == 0 while a positive value appeared in the window. Examples include TCP CLOSE/FIN wait metrics on Tomcat03 and Tomcat02, and multiple MySQL-related counters (timestamps around 2021-03-23 18:00:00+08:00). These indicate previously-zero baselines that observed nonzero events during the incident window.\n"
    f"- Elevated CPU wait (OSLinux-CPU_CPU_CPUWio) on multiple hosts with large breach ratios:\n"
    f"  * Redis02: count=2, earliest=2021-03-23 17:52:00+08:00, max_value=1.2090, breach_ratio≈6.70\n"
    f"  * MG02:   count=2, earliest=2021-03-23 17:55:00+08:00, max_value=4.6500, breach_ratio≈5.12\n"
    f"  * Tomcat03: count=2, earliest=2021-03-23 17:53:00+08:00, max_value=3.7262, breach_ratio≈4.71\n"
    f"  * Tomcat02: count=2, earliest=2021-03-23 17:55:00+08:00, max_value=2.4798, breach_ratio≈4.13\n"
    f"  * Tomcat04: count=2, earliest=2021-03-23 17:45:00+08:00, max_value=1.6218, breach_ratio≈2.90\n"
    f"- Several disk-related metrics (DSKRead/DSKBps/DSKRTps) show ~2.33x breaches on Tomcat and Redis hosts (single-point breaches with timestamps around 17:30–17:55+08:00).\n"
    f"- A notable MySQL metric 'max trx lock memory bytes' on Mysql01 breached from baseline 0 to max_value=1136 at 2021-03-23 18:00:00+08:00 (infinite breach ratio). This and related MySQL counters at the same timestamp merit attention.\n\n"
    f"Interpretation and next steps:\n"
    f"- The strongest signals implicate transient CPU wait (I/O wait) spikes on Redis/MG/Tomcat hosts and isolated disk/IO bursts on several Tomcat/Redis nodes; these could point to disk I/O pressure or blocking I/O causing increased CPU WIO.\n"
    f"- The infinite-ratio anomalies (baseline 0 → positive) for TCP CLOSE/FIN wait and MySQL counters indicate new events during the window and should be inspected (logs/traces) for connection or DB contention issues.\n"
    f"- Recommended immediate actions: check system logs and database slow/lock traces around 2021-03-23 17:30–18:00+08:00 for Tomcat02/03/04, Redis02, MG02, and Mysql01/Mysql02; investigate disk I/O and MySQL lock metrics.\n\n"
    f"Note: full intermediate tables (metric_p95_df, metric_window_df, metric_anomaly_points, anomaly_agg_all, anomaly_agg_top20) are retained in the kernel for further inspection."
)

summary 

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name    global_P95
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.967656e+01
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.141293e+08
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07
4         IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  4.557100e-01
...        ...                                                ...           ...
1106  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  0.000000e+00
1107  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  1.055000e+01
1108  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  0.000000e+00
1109  apache02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.200000e+01
1110  apache02       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.770360e+01

[1111 rows x 3 columns], (21161, 6), (583, 6), (441, 7),      cmdb_id                                           kpi_name  global_P95  count_anom_points   earliest_anom_timestamp  max_value  breach_ratio
0   Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.000000                  1 2021-03-23 17:32:00+08:00     1.0000           inf
1   Tomcat03       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.000000                  1 2021-03-23 17:42:00+08:00     1.0000           inf
2   Tomcat02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.000000                  1 2021-03-23 17:44:00+08:00     1.0000           inf
3    Mysql01               Mysql-MySQL_3306_Max trx rows locked    0.000000                  1 2021-03-23 18:00:00+08:00     1.0000           inf
4    Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified    0.000000                  1 2021-03-23 18:00:00+08:00     1.0000           inf
5    Mysql01         Mysql-MySQL_3306_max trx lock memory bytes    0.000000                  1 2021-03-23 18:00:00+08:00  1136.0000           inf
6    Mysql02        Mysql-MySQL_3306_Innodb data pending writes    0.000000                  1 2021-03-23 17:55:00+08:00     1.0000           inf
7   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    0.000000                  1 2021-03-23 17:33:00+08:00     1.0000           inf
8    Redis02                             OSLinux-CPU_CPU_CPUWio    0.157023                  2 2021-03-23 17:52:00+08:00     1.2090      6.699534
9       MG02                             OSLinux-CPU_CPU_CPUWio    0.759717                  2 2021-03-23 17:55:00+08:00     4.6500      5.120701
10  Tomcat03                             OSLinux-CPU_CPU_CPUWio    0.652714                  2 2021-03-23 17:53:00+08:00     3.7262      4.708781
11  Tomcat02                             OSLinux-CPU_CPU_CPUWio    0.483534                  2 2021-03-23 17:55:00+08:00     2.4798      4.128494
12  Tomcat04                             OSLinux-CPU_CPU_CPUWio    0.415868                  2 2021-03-23 17:45:00+08:00     1.6218      2.899797
13   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.120000                  1 2021-03-23 17:35:00+08:00     0.4000      2.333333
14  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.054990                  1 2021-03-23 17:31:00+08:00     0.1833      2.333333
15  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    5.840010                  1 2021-03-23 17:45:00+08:00    19.4667      2.333333
16  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000                  1 2021-03-23 17:39:00+08:00     1.0000      2.333333
17  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    1.800000                  1 2021-03-23 17:31:00+08:00     6.0000      2.333333
18  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    1.800000                  1 2021-03-23 17:55:00+08:00     6.0000      2.333333
19   Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000                  1 2021-03-23 17:30:00+08:00     1.0000      2.333333)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing kernel variables: metric_anomaly_points, metric_window_df, metric_p95_df
tz = pytz.timezone('Asia/Shanghai')

# Prepare anomaly points table (ensure expected columns exist)
if 'metric_anomaly_points' not in globals():
    # fallback: try to reconstruct anomalies from previous variables if available
    raise RuntimeError("metric_anomaly_points not found in kernel. Please run prior anomaly detection step.")

anoms = metric_anomaly_points.copy()
# Ensure necessary columns and types
for col in ['cmdb_id', 'kpi_name', 'dt', 'value', 'global_P95']:
    if col not in anoms.columns:
        raise RuntimeError(f"Expected column '{col}' in metric_anomaly_points")

anoms = anoms[['cmdb_id', 'kpi_name', 'dt', 'value', 'global_P95']].sort_values(['cmdb_id', 'kpi_name', 'dt']).reset_index(drop=True)

if anoms.shape[0] == 0:
    # No anomalies found: create empty fault_runs and return empty top table
    fault_runs = pd.DataFrame(columns=['cmdb_id','kpi_name','start_timestamp','end_timestamp','n_points','global_P95','max_value','breach_ratio'])
    fault_runs_top20 = fault_runs.copy()
else:
    # Compute inter-point gaps in seconds per (cmdb_id, kpi_name)
    anoms['time_diff_s'] = anoms.groupby(['cmdb_id', 'kpi_name'])['dt'].diff().dt.total_seconds().fillna(0)

    # Mark new run when gap > 60 seconds
    anoms['new_run_flag'] = (anoms['time_diff_s'] > 60).astype(int)

    # Run index within each group: cumulative sum of new_run_flag per pair
    anoms['run_idx'] = anoms.groupby(['cmdb_id', 'kpi_name'])['new_run_flag'].cumsum()

    # Create run identifier by combining keys
    # We'll aggregate by cmdb_id, kpi_name, run_idx
    grp = anoms.groupby(['cmdb_id', 'kpi_name', 'run_idx'], as_index=False)

    fault_runs = grp.agg(
        start_timestamp=('dt', 'min'),
        end_timestamp=('dt', 'max'),
        n_points=('value', 'size'),
        max_value=('value', 'max'),
        global_P95=('global_P95', 'first')
    )

    # Compute breach_ratio with proper handling for global_P95 == 0
    def compute_breach(row):
        gp = row['global_P95']
        mv = row['max_value']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            if mv > 0:
                return np.inf
            else:
                return 0.0
        return (mv - gp) / gp

    fault_runs['breach_ratio'] = fault_runs.apply(compute_breach, axis=1)

    # Filter out likely-noise runs:
    # Keep runs where n_points >= 2
    # Keep single-point runs only if breach_ratio > 0.5 OR (global_P95 == 0 and max_value > 0)
    cond_keep_multi = fault_runs['n_points'] >= 2
    cond_keep_single = (
        (fault_runs['n_points'] == 1) &
        (
            (fault_runs['breach_ratio'] > 0.5) |
            ((fault_runs['global_P95'] == 0) & (fault_runs['max_value'] > 0))
        )
    )
    fault_runs = fault_runs[cond_keep_multi | cond_keep_single].reset_index(drop=True)

    # Sort by breach_ratio descending and keep top 20 for compact display later
    fault_runs = fault_runs.sort_values('breach_ratio', ascending=False).reset_index(drop=True)

    # Ensure timestamp columns are timezone-aware (should already be)
    # Store full runs in kernel variable 'fault_runs' as required

# Prepare compact top-20 table for display with requested columns
fault_runs_top20 = fault_runs.head(20)[['cmdb_id','kpi_name','start_timestamp','end_timestamp','n_points','global_P95','max_value','breach_ratio']]

# Display compact result (per environment rules, return variable)
fault_runs_top20

```
Out[3]:
```
Summary of detected fault runs (incident window 2021-03-23 17:30–18:00 UTC+8):

- Overall: I segmented consecutive anomalous points per (cmdb_id, kpi_name) into "fault runs" and filtered noise. The result contains up to 20 top runs ranked by breach ratio.

- Most prominent multi-point run:
  - MG02, OSLinux-CPU_CPU_CPUWio: 2 points from 2021-03-23 17:55 → 17:56, max_value=4.65, global_P95≈0.7597, breach_ratio≈5.12. This indicates a sustained I/O-wait spike on MG02.

- Other strong CPU I/O-wait signals (mostly single-point or short):
  - Redis02, OSLinux-CPU_CPU_CPUWio: 1 point at 17:55, max=1.2090, global_P95≈0.1570, breach_ratio≈6.70.
  - Tomcat03, Tomcat02, Tomcat04 also show elevated CPU WIO (single-point runs with breach_ratios between ~2.9–4.7).

- Database and connection events (single-point but high/novel values):
  - Mysql01 shows several MySQL-related counters at 18:00 with baseline 0 → positive values (e.g., "max trx lock memory bytes" = 1136) — these produce infinite breach_ratio (baseline was 0). These indicate new/rare DB lock or contention events at the incident end time.
  - Mysql02 shows a single-point "Innodb data pending writes" = 1 at 17:55 with baseline 0 (infinite breach).

- Network and TCP anomalies:
  - TCP CLOSE-WAIT / FIN-WAIT metrics on Tomcat02/Tomcat03 appear as single-point runs (baseline 0 → 1) — these are novel connection states during the window and may reflect connection handling/problems.

- Disk I/O bursts:
  - Several hosts (Tomcat01/02/03, Redis02, MG01) show single-point disk metrics (DSKRead/DSKBps/DSKRTps) with ~2.33× breaches over their P95 baselines. Example: Tomcat02 DSKRead at 17:45 jumped to 19.4667 from a P95 ~5.84.

Interpretation and recommended next steps:
- The strongest signals are short-lived I/O-wait spikes across multiple hosts (MG02, Redis02, Tomcats) and isolated disk I/O bursts on Tomcat/Redis nodes — these point toward transient disk I/O pressure causing CPU I/O-wait.
- Concurrent novel MySQL lock/transaction metrics at 18:00 on Mysql01/Mysql02 indicate DB contention or locking events that deserve immediate inspection.
- TCP CLOSE/FIN new events suggest connection-level issues on Tomcat hosts; check application logs and connection handling around the timestamps.
- Actionable triage: inspect system logs, MySQL slow/lock traces, disk IO metrics and controller logs for Tomcat02/03/04, Redis02, MG02, and Mysql01 around 17:30–18:00+08:00. Correlate with application traces and error logs for root cause confirmation.

The full runs table is stored in the kernel variable 'fault_runs' for further inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           start_timestamp             end_timestamp  n_points  global_P95  max_value  breach_ratio
0   Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-23 17:33:00+08:00 2021-03-23 17:33:00+08:00         1    0.000000     1.0000           inf
1    Mysql02        Mysql-MySQL_3306_Innodb data pending writes 2021-03-23 17:55:00+08:00 2021-03-23 17:55:00+08:00         1    0.000000     1.0000           inf
2    Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified 2021-03-23 18:00:00+08:00 2021-03-23 18:00:00+08:00         1    0.000000     1.0000           inf
3    Mysql01         Mysql-MySQL_3306_max trx lock memory bytes 2021-03-23 18:00:00+08:00 2021-03-23 18:00:00+08:00         1    0.000000  1136.0000           inf
4    Mysql01               Mysql-MySQL_3306_Max trx rows locked 2021-03-23 18:00:00+08:00 2021-03-23 18:00:00+08:00         1    0.000000     1.0000           inf
5   Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-23 17:32:00+08:00 2021-03-23 17:32:00+08:00         1    0.000000     1.0000           inf
6   Tomcat03       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 2021-03-23 17:42:00+08:00 2021-03-23 17:42:00+08:00         1    0.000000     1.0000           inf
7   Tomcat02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 2021-03-23 17:44:00+08:00 2021-03-23 17:44:00+08:00         1    0.000000     1.0000           inf
8    Redis02                             OSLinux-CPU_CPU_CPUWio 2021-03-23 17:55:00+08:00 2021-03-23 17:55:00+08:00         1    0.157023     1.2090      6.699534
9       MG02                             OSLinux-CPU_CPU_CPUWio 2021-03-23 17:55:00+08:00 2021-03-23 17:56:00+08:00         2    0.759717     4.6500      5.120701
10  Tomcat03                             OSLinux-CPU_CPU_CPUWio 2021-03-23 17:55:00+08:00 2021-03-23 17:55:00+08:00         1    0.652714     3.7262      4.708781
11  Tomcat02                             OSLinux-CPU_CPU_CPUWio 2021-03-23 17:55:00+08:00 2021-03-23 17:55:00+08:00         1    0.483534     2.4798      4.128494
12  Tomcat04                             OSLinux-CPU_CPU_CPUWio 2021-03-23 17:55:00+08:00 2021-03-23 17:55:00+08:00         1    0.415868     1.6218      2.899797
13  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-23 17:31:00+08:00 2021-03-23 17:31:00+08:00         1    0.054990     0.1833      2.333333
14  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead 2021-03-23 17:45:00+08:00 2021-03-23 17:45:00+08:00         1    5.840010    19.4667      2.333333
15   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer... 2021-03-23 17:35:00+08:00 2021-03-23 17:35:00+08:00         1    0.120000     0.4000      2.333333
16  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-23 17:55:00+08:00 2021-03-23 17:55:00+08:00         1    1.800000     6.0000      2.333333
17      MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-23 18:00:00+08:00 2021-03-23 18:00:00+08:00         1    0.300000     1.0000      2.333333
18  Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps 2021-03-23 17:45:00+08:00 2021-03-23 17:45:00+08:00         1    0.140010     0.4667      2.333333
19  Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps 2021-03-23 17:39:00+08:00 2021-03-23 17:39:00+08:00         1    0.300000     1.0000      2.333333```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing kernel variables where possible: trace_df, components (if not present, define)
tz = pytz.timezone('Asia/Shanghai')

try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Reuse components list if available, else define
try:
    components
except NameError:
    components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                  "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Ensure dt column (timezone-aware UTC+8) exists
if 'dt' not in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to components of interest
trace_df_filtered = trace_df[trace_df['cmdb_id'].isin(components)].copy()

# Ensure numeric 'value'
trace_df_filtered['value'] = pd.to_numeric(trace_df_filtered['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full filtered series (no time filtering)
def p95(arr):
    a = np.asarray(arr.dropna())
    return float(np.nanpercentile(a, 95)) if a.size > 0 else np.nan

trace_p95_df = (
    trace_df_filtered
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(global_P95=('value', p95))
)

# 2) Filter trace rows to the incident window (UTC+8)
start_dt = tz.localize(pd.to_datetime("2021-03-23 17:30:00"))
end_dt   = tz.localize(pd.to_datetime("2021-03-23 18:00:00"))

trace_window_df = trace_df_filtered[
    (trace_df_filtered['dt'] >= start_dt) & (trace_df_filtered['dt'] <= end_dt)
].copy()

# 3) Identify anomalies where value > global_P95
# Merge global P95 into windowed traces
trace_window_df = trace_window_df.merge(trace_p95_df, on=['cmdb_id', 'trace_name'], how='left')

trace_anomalies = trace_window_df[
    trace_window_df['global_P95'].notna() &
    trace_window_df['value'].notna() &
    (trace_window_df['value'] > trace_window_df['global_P95'])
].copy()

# Aggregate anomalies per pair
if trace_anomalies.shape[0] == 0:
    trace_anomaly_agg = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','count_anom_points',
                                              'earliest_anom_timestamp','max_value','breach_ratio'])
else:
    trace_anomaly_agg = (
        trace_anomalies
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            global_P95 = ('global_P95', 'first'),
            count_anom_points = ('value', 'size'),
            earliest_anom_timestamp = ('dt', 'min'),
            max_value = ('value', 'max')
        )
    )
    # breach_ratio, handling global_P95 == 0
    def br(row):
        gp = row['global_P95']
        mv = row['max_value']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            return np.inf if mv > 0 else 0.0
        return (mv - gp) / gp

    trace_anomaly_agg['breach_ratio'] = trace_anomaly_agg.apply(br, axis=1)

# 4) Sort by breach_ratio desc and keep top 20; store full anomaly points table in kernel variable 'trace_anomalies'
trace_anomaly_top20 = trace_anomaly_agg.sort_values('breach_ratio', ascending=False).head(20).reset_index(drop=True)

# Expose results: trace_anomalies (full points) and compact top20 table
trace_anomalies, trace_anomaly_top20

```
Out[4]:
```
# Plain-English summary using kernel-stored results (display as a string)
summary = (
    "Trace anomaly detection summary (components restricted to the provided list, window 2021-03-23 17:30→18:00 UTC+8):\n"
    "- Result: No trace points exceeded their pair-wise global P95 baselines in the incident window.\n"
    "- Both the per-point table (trace_anomalies) and the aggregated anomaly table are empty.\n"
    "- Interpretation: trace-level features (duration/p95/row_count) did not show anomalous spikes for the listed components during that window.\n"
    "- Suggested next steps: confirm trace coverage/sampling for the components and time range; if coverage is sparse, widen the time window or correlate with metric/log anomalies for root-cause investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value, dt, global_P95]
Index: [], Empty DataFrame
Columns: [cmdb_id, trace_name, global_P95, count_anom_points, earliest_anom_timestamp, max_value, breach_ratio]
Index: [])```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd
import pytz

# Reuse kernel variables if available
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

try:
    components
except NameError:
    components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                  "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

tz = pytz.timezone('Asia/Shanghai')

# Ensure timezone-aware datetime column 'dt' in UTC+8
if 'dt' not in log_df.columns:
    log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter to components of interest
log_df_filtered = log_df[log_df['cmdb_id'].isin(components)].copy()

# Ensure numeric 'value'
log_df_filtered['value'] = pd.to_numeric(log_df_filtered['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, log_name) using full series (no time filter)
def p95(arr):
    a = np.asarray(arr.dropna())
    return float(np.nanpercentile(a, 95)) if a.size > 0 else np.nan

log_p95_df = (
    log_df_filtered
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(global_P95=('value', p95))
)

# 2) Filter log rows to the incident window (UTC+8)
start_dt = tz.localize(pd.to_datetime("2021-03-23 17:30:00"))
end_dt   = tz.localize(pd.to_datetime("2021-03-23 18:00:00"))

log_window_df = log_df_filtered[
    (log_df_filtered['dt'] >= start_dt) & (log_df_filtered['dt'] <= end_dt)
].copy()

# 3) Identify anomalies: value > global_P95 OR (global_P95 == 0 AND value > 0)
log_window_df = log_window_df.merge(log_p95_df, on=['cmdb_id', 'log_name'], how='left')

# Keep rows where global_P95 exists (not NaN)
cond_anom = (
    log_window_df['global_P95'].notna() &
    log_window_df['value'].notna() &
    (
        (log_window_df['value'] > log_window_df['global_P95']) |
        ((log_window_df['global_P95'] == 0) & (log_window_df['value'] > 0))
    )
)

log_anomalies = log_window_df[cond_anom].copy()  # full anomaly points stored in kernel variable

# Aggregate anomalies per (cmdb_id, log_name)
if log_anomalies.shape[0] == 0:
    log_anomaly_agg = pd.DataFrame(columns=['cmdb_id','log_name','global_P95','count_anom_points',
                                            'earliest_anom_timestamp','max_value','breach_ratio'])
else:
    log_anomaly_agg = (
        log_anomalies
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(
            global_P95 = ('global_P95', 'first'),
            count_anom_points = ('value', 'size'),
            earliest_anom_timestamp = ('dt', 'min'),
            max_value = ('value', 'max')
        )
    )

    # Compute breach_ratio with handling for global_P95 == 0
    def compute_breach(row):
        gp = row['global_P95']
        mv = row['max_value']
        if pd.isna(gp):
            return np.nan
        if gp == 0:
            return np.inf if mv > 0 else 0.0
        return (mv - gp) / gp

    log_anomaly_agg['breach_ratio'] = log_anomaly_agg.apply(compute_breach, axis=1)

# 4) Top 20 by breach_ratio desc, store full anomaly points in 'log_anomalies'
log_anomaly_top20 = log_anomaly_agg.sort_values('breach_ratio', ascending=False).head(20).reset_index(drop=True)

# Return only the compact table (full points kept in 'log_anomalies' variable)
log_anomaly_top20

```
Out[5]:
```
Summary of log anomalies within 2021-03-23 17:30–18:00 (components restricted to the provided list):

- Overview:
  - Detected modest log anomalies across several hosts; no zero→positive (infinite breach) cases in logs.
  - Most anomalies are small relative increases in log row counts and slight increases in GC pause durations.

- Top findings (high-level):
  - Mysql01, log.row_count: global P95 = 10.7 → observed max = 16 at 2021-03-23 17:30 (2 anomalous points, breach_ratio ≈ 0.495). This is the largest log-related breach but still below a 50% relative jump.
  - Tomcat GC pauses: small increases on Tomcat03 (breach ≈ 0.163 at 17:36), Tomcat02 (≈ 0.132 at 17:40), Tomcat04 (≈ 0.081 at 17:33). These are very small absolute GC pause increases.
  - Multiple hosts (apache01, apache02, Tomcat01–04) show short two-point increases in log.row_count around 17:30 with low breach ratios (≈0.03–0.044), indicating slightly higher log rates at the start of the window.

- Interpretation and recommended next steps:
  - The log anomalies are generally mild. The most notable is the Mysql01 row_count increase at 17:30—investigate MySQL logs/queries around that time and correlate with the metric-level MySQL lock/transaction anomalies seen at ~18:00.
  - Small GC pause increases on Tomcat hosts are unlikely to be the primary cause but worth checking application logs for any related spikes or frequent minor GCs.
  - Correlate these log anomalies with the metric anomalies (CPU I/O-wait, disk bursts, and MySQL lock events) and with traces/errors to determine if log changes reflect meaningful service impact.

The full per-point anomaly table is stored in the kernel variable 'log_anomalies' for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name  global_P95  count_anom_points   earliest_anom_timestamp  max_value  breach_ratio
0   Mysql01     log.row_count     10.7000                  2 2021-03-23 17:30:00+08:00      16.00      0.495327
1  Tomcat03  log.gc_pause_max      0.0860                  1 2021-03-23 17:36:00+08:00       0.10      0.162791
2  Tomcat02  log.gc_pause_max      0.1060                  1 2021-03-23 17:40:00+08:00       0.12      0.132075
3  Tomcat01     log.row_count    388.0000                  2 2021-03-23 17:30:00+08:00     432.00      0.113402
4  Tomcat04  log.gc_pause_max      0.0925                  1 2021-03-23 17:33:00+08:00       0.10      0.081081
5  apache02     log.row_count    761.5000                  2 2021-03-23 17:30:00+08:00     795.00      0.043992
6  apache01     log.row_count    770.0000                  2 2021-03-23 17:30:00+08:00     796.00      0.033766
7  Tomcat02     log.row_count    398.0000                  2 2021-03-23 17:30:00+08:00     411.00      0.032663
8  Tomcat03     log.row_count    366.5000                  2 2021-03-23 17:30:00+08:00     377.00      0.028649
9  Tomcat04     log.row_count    390.0000                  2 2021-03-23 17:31:00+08:00     397.00      0.017949```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure 'fault_runs' exists in kernel
try:
    fault_runs
except NameError:
    fault_runs = pd.DataFrame(columns=['cmdb_id','kpi_name','start_timestamp','end_timestamp',
                                       'n_points','global_P95','max_value','breach_ratio'])

# If empty, return empty summary with requested columns
if fault_runs.shape[0] == 0:
    summary_df = pd.DataFrame(columns=[
        'cmdb_id','total_fault_runs','earliest_fault_start_timestamp',
        'latest_fault_end_timestamp','max_breach_ratio','median_breach_ratio'
    ])
else:
    # Ensure timestamp columns are datetime and timezone-aware (Asia/Shanghai)
    for col in ['start_timestamp', 'end_timestamp']:
        if not pd.api.types.is_datetime64_any_dtype(fault_runs[col]):
            fault_runs[col] = pd.to_datetime(fault_runs[col])
        # localize/convert to Asia/Shanghai if tz-naive
        if fault_runs[col].dt.tz is None:
            fault_runs[col] = fault_runs[col].dt.tz_localize(tz)
        else:
            fault_runs[col] = fault_runs[col].dt.tz_convert(tz)

    # Ensure breach_ratio numeric
    fault_runs['breach_ratio'] = pd.to_numeric(fault_runs['breach_ratio'], errors='coerce')

    # Group by component and compute aggregates
    def median_breach(series):
        vals = series.values
        # If any finite values exist, compute median of finite values
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size > 0:
            return float(np.median(finite_vals))
        # If no finite but any positive infinite, return inf
        if np.isposinf(vals).any():
            return np.inf
        # If no finite and no posinf but neginf etc, return NaN
        return np.nan

    agg = fault_runs.groupby('cmdb_id').agg(
        total_fault_runs = ('kpi_name', 'size'),
        earliest_fault_start_timestamp = ('start_timestamp', 'min'),
        latest_fault_end_timestamp = ('end_timestamp', 'max'),
        max_breach_ratio = ('breach_ratio', 'max')
    ).reset_index()

    # compute median separately using groupby apply
    med = fault_runs.groupby('cmdb_id')['breach_ratio'].apply(median_breach).reset_index(name='median_breach_ratio')

    summary_df = agg.merge(med, on='cmdb_id', how='left')

    # Sort by total_fault_runs desc, then max_breach_ratio desc
    summary_df = summary_df.sort_values(['total_fault_runs','max_breach_ratio'], ascending=[False, False]).reset_index(drop=True)

    # Keep only top 20
    summary_df = summary_df.head(20)

# Ensure compact display: select and order columns
summary_df = summary_df[[
    'cmdb_id','total_fault_runs','earliest_fault_start_timestamp',
    'latest_fault_end_timestamp','max_breach_ratio','median_breach_ratio'
]]

summary_df

```
Out[6]:
```
Summary — per-component fault-run overview (top components):

- Mysql02: 46 fault runs (earliest 2021-03-23 17:41, latest 17:59). max_breach_ratio = inf (some runs jumped from baseline 0 → positive), median_breach_ratio ≈ 0.000048. Highest count of runs; investigate new DB events/locks and why many short runs occurred.

- Mysql01: 40 fault runs (earliest 17:58, latest 18:00). max_breach_ratio = inf (zero→positive events at ~18:00), median_breach_ratio ≈ 0.000041. Strong signal of novel MySQL lock/transaction activity at the incident end.

- MG02: 14 fault runs (17:30 → 18:00). max_breach_ratio ≈ 5.12, median ≈ 0.86. Notable sustained I/O-wait spikes (high breach ratios) — prioritize I/O and disk wait investigation.

- Tomcat01: 13 fault runs (17:43 → 18:00). max_breach_ratio ≈ 2.33, median ≈ 1.09. Multiple runs with moderate-to-high breach ratios — check thread/IO/disk metrics and application logs.

- Tomcat02: 11 fault runs (17:30 → 18:00). max_breach_ratio = inf, median ≈ 0.538. Contains zero→positive anomalies (TCP/connection events) and disk/IO bursts — inspect connection handling and disk I/O.

- Tomcat03: 10 fault runs (17:31 → 18:00). max_breach_ratio = inf, median ≈ 0.737. Similar pattern to Tomcat02; check TCP states and disk/IO behavior.

- Redis02: 9 fault runs (17:30 → 18:00). max_breach_ratio ≈ 6.70, median ≈ 1.42. Large CPU I/O-wait spikes observed — strong candidate for root cause related to disk or blocking I/O.

- Tomcat04: 7 fault runs (17:41 → 18:00). max_breach_ratio ≈ 2.90, median = 1.00. Shows CPU WIO / disk-related breaches.

- MG01: 7 fault runs (17:45 → 18:00). max_breach_ratio ≈ 2.33, median ≈ 0.54.

Interpretation and recommended next steps:
- High priority: Mysql02 and Mysql01 (many runs and infinite breach ratios) — investigate MySQL locks, transactions, and slow queries around 17:58–18:00.
- Next: Redis02 and MG02 (high max and median breach ratios) — investigate disk I/O, filesystem latency, and processes causing high I/O wait.
- Tomcat hosts (01–04) show multiple runs and some zero→positive anomalies (TCP states, disk reads) — correlate with application logs and connection traces.
- Triage actions: check DB lock/slow logs, check disk I/O metrics and queueing, inspect OS/kernel logs and application logs for the listed hosts during 17:30–18:00 UTC+8.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_fault_runs earliest_fault_start_timestamp latest_fault_end_timestamp  max_breach_ratio  median_breach_ratio
0   Mysql02                46      2021-03-23 17:41:00+08:00  2021-03-23 17:59:00+08:00               inf             0.000048
1   Mysql01                40      2021-03-23 17:58:00+08:00  2021-03-23 18:00:00+08:00               inf             0.000041
2      MG02                14      2021-03-23 17:30:00+08:00  2021-03-23 18:00:00+08:00          5.120701             0.862150
3  Tomcat01                13      2021-03-23 17:43:00+08:00  2021-03-23 18:00:00+08:00          2.333333             1.094241
4  Tomcat02                11      2021-03-23 17:30:00+08:00  2021-03-23 18:00:00+08:00               inf             0.538462
5  Tomcat03                10      2021-03-23 17:31:00+08:00  2021-03-23 18:00:00+08:00               inf             0.737069
6   Redis02                 9      2021-03-23 17:30:00+08:00  2021-03-23 18:00:00+08:00          6.699534             1.424242
7  Tomcat04                 7      2021-03-23 17:41:00+08:00  2021-03-23 18:00:00+08:00          2.899797             1.000000
8      MG01                 7      2021-03-23 17:45:00+08:00  2021-03-23 18:00:00+08:00          2.333333             0.538462```
```